# Notebook v3 — Regenerar Ilustraciones 17, 19 y 20 con n=6.596

**Para:** María José Morte Ruiz

## Cambios respecto a v2

- Carga `data/06_evaluacion/meta_test.parquet` para tener `curso_aca_ini` por alumno y filtrar cohorte 2009.
- Regenera SHAP Top-15 sobre n=6.596.
- Regenera curva fiabilidad sobre n=6.596.
- Carga directamente `data/06_interpretacion/robustez/auc_por_cohorte.parquet` para Ilustración 19.

## Cómo usar

1. Sustituye el v2 por este v3 en `c:\FF\AU_UJI_v2\`.
2. Run All.
3. Sube los 4 JSON generados en `output_para_claude/`.

In [1]:
# === CELDA 1: Imports y rutas ===
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ROOT = Path.cwd()
while not (ROOT / 'src').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("No encontrado src/")
    ROOT = ROOT.parent

print(f"ROOT: {ROOT}")

OUTPUT_DIR = ROOT / 'output_para_claude'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

ROOT: c:\FF\AU_UJI_v2
Output: c:\FF\AU_UJI_v2\output_para_claude


In [2]:
# === CELDA 2: Cargar artefactos + meta_test con cohorte ===
DIR_MOD = ROOT / 'data' / '05_modelado'
DIR_MODELS = DIR_MOD / 'models'
DIR_EVAL = ROOT / 'data' / '06_evaluacion'

pipeline_prep = joblib.load(DIR_MOD / 'pipeline_preprocesamiento.pkl')
modelo = joblib.load(DIR_MODELS / 'LightGBM__none.pkl')

X_test_prep = pd.read_parquet(DIR_MOD / 'X_test_prep.parquet')
y_test = pd.read_parquet(DIR_MOD / 'y_test.parquet').squeeze()
X_test = pd.read_parquet(DIR_MOD / 'X_test.parquet')

# Meta test con cohorte
meta_test = pd.read_parquet(DIR_EVAL / 'meta_test.parquet')

print(f"X_test_prep: {X_test_prep.shape}")
print(f"y_test:      {y_test.shape}")
print(f"meta_test:   {meta_test.shape}")
print(f"\nColumnas meta_test: {meta_test.columns.tolist()}")
print(f"\nDistribución cohorte:")
print(meta_test['curso_aca_ini'].value_counts().sort_index())

X_test_prep: (6725, 27)
y_test:      (6725,)
meta_test:   (6725, 14)

Columnas meta_test: ['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'curso_aca', 'vive_fuera', 'cupo', 'n_titulaciones', 'flag_cautela']

Distribución cohorte:
curso_aca_ini
2009    129
2010    657
2011    715
2012    680
2013    603
2014    633
2015    580
2016    539
2017    562
2018    538
2019    505
2020    584
Name: count, dtype: Int64


In [3]:
# === CELDA 3: Filtrar a n=6.596 (cohortes >= 2010) ===
# meta_test tiene 6725 filas alineadas con X_test_prep / y_test
assert len(meta_test) == len(X_test_prep) == len(y_test), 'Tamaños no coinciden'

# Máscara: cohorte >= 2010
mask_2010 = meta_test['curso_aca_ini'] >= 2010
n_filtrados = (~mask_2010).sum()
n_quedan = mask_2010.sum()

print(f"Total test: {len(meta_test)}")
print(f"Filtrados (cohorte 2009): {n_filtrados}")
print(f"Quedan (cohorte >=2010): {n_quedan}")
print()

# Aplicar máscara conservando alineación posicional
X_test_prep_2010 = X_test_prep.loc[mask_2010.values].copy()
y_test_2010 = y_test.loc[mask_2010.values].copy() if hasattr(y_test, 'loc') else y_test[mask_2010.values]
meta_test_2010 = meta_test.loc[mask_2010.values].copy()

print(f"X_test_prep_2010: {X_test_prep_2010.shape}")
print(f"y_test_2010:      {y_test_2010.shape}")
print(f"Tasa abandono filtrada: {y_test_2010.mean():.4f}")

Total test: 6725
Filtrados (cohorte 2009): 129
Quedan (cohorte >=2010): 6596

X_test_prep_2010: (6596, 27)
y_test_2010:      (6596,)
Tasa abandono filtrada: 0.2955


In [4]:
# === CELDA 4: Predicciones sobre n=6.596 ===
if hasattr(modelo, 'named_steps') and 'model' in modelo.named_steps:
    modelo_real = modelo.named_steps['model']
    print("Modelo es Pipeline")
else:
    modelo_real = modelo
    print("Modelo es estimator directo")

y_proba_2010 = modelo.predict_proba(X_test_prep_2010)[:, 1]
print(f"Predicciones: {y_proba_2010.shape}")
print(f"Rango: [{y_proba_2010.min():.4f}, {y_proba_2010.max():.4f}]")

Modelo es Pipeline
Predicciones: (6596,)
Rango: [0.0003, 0.9970]


In [5]:
# === CELDA 5: Ilustración 17 — SHAP Top-15 sobre n=6.596 ===
import shap

explainer = shap.TreeExplainer(modelo_real)
shap_values = explainer.shap_values(X_test_prep_2010)

if isinstance(shap_values, list):
    shap_array = shap_values[1]
else:
    shap_array = shap_values

importance = np.abs(shap_array).mean(axis=0)
feature_names = X_test_prep_2010.columns.tolist()

df_imp = pd.DataFrame({'feature': feature_names, 'importance': importance})
df_imp = df_imp.sort_values('importance', ascending=False).head(15).reset_index(drop=True)

print("Top-15 features SHAP (n=6.596):")
print(df_imp.to_string())

ilustracion_17 = {
    'features': df_imp['feature'].tolist(),
    'importance': df_imp['importance'].round(4).tolist(),
    'n_observaciones': int(len(X_test_prep_2010))
}
with open(OUTPUT_DIR / 'ilustracion_17_shap_top15.json', 'w', encoding='utf-8') as f:
    json.dump(ilustracion_17, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {OUTPUT_DIR / 'ilustracion_17_shap_top15.json'}")

Top-15 features SHAP (n=6.596):
                     feature  importance
0    cred_superados_anio_1er    0.728107
1         n_anios_trabajando    0.697225
2               n_anios_beca    0.630415
3             cred_repetidos    0.448663
4             anios_sin_beca    0.418735
5          situacion_laboral    0.416546
6              nota_1er_anio    0.368388
7          n_anios_sin_notas    0.345085
8                nota_acceso    0.207781
9   tasa_abandono_titulacion    0.132226
10           tasa_repeticion    0.124582
11              edad_entrada    0.123765
12                 max_pagos    0.084990
13                      rama    0.068488
14                 provincia    0.052760

Guardado: c:\FF\AU_UJI_v2\output_para_claude\ilustracion_17_shap_top15.json


C:\Users\mjmor\anaconda3\envs\tfm_abandono\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [6]:
# === CELDA 6: Diccionario nombres legibles ===
FEATURES_DICT_PROPUESTO = {
    'cred_superados_anio_1er': 'Créditos superados en 1er año',
    'n_anios_trabajando': 'Años trabajando',
    'n_anios_beca': 'Años con beca',
    'cred_repetidos': 'Créditos repetidos',
    'anios_sin_beca': 'Años sin beca',
    'situacion_laboral': 'Situación laboral',
    'nota_1er_anio': 'Nota media 1er año',
    'n_anios_sin_notas': 'Años sin calificaciones',
    'nota_acceso': 'Nota de acceso',
    'tasa_abandono_titulacion': 'Tasa abandono titulación',
    'tasa_repeticion': 'Tasa de repetición',
    'edad_entrada': 'Edad de entrada',
    'max_pagos': 'Pagos máximos',
    'rama': 'Rama de conocimiento',
    'provincia': 'Provincia',
    'via_acceso': 'Vía de acceso',
    'cupo': 'Cupo',
    'sexo': 'Sexo',
    'pais_nombre': 'País',
    'universidad_origen': 'Universidad de origen',
    'indicador_interrupcion': 'Indicador interrupción',
    'orden_preferencia': 'Orden de preferencia',
    'anios_gap': 'Años gap',
    'nota_selectividad': 'Nota selectividad',
    'nota_1er_anio_missing': 'Nota 1er año (missing)',
    'nota_acceso_missing': 'Nota acceso (missing)',
    'nota_selectividad_missing': 'Nota selectividad (missing)',
}

with open(OUTPUT_DIR / 'features_dict_legible.json', 'w', encoding='utf-8') as f:
    json.dump(FEATURES_DICT_PROPUESTO, f, indent=2, ensure_ascii=False)
print(f"Guardado: {OUTPUT_DIR / 'features_dict_legible.json'}")

Guardado: c:\FF\AU_UJI_v2\output_para_claude\features_dict_legible.json


In [7]:
# === CELDA 7: Ilustración 19 — AUC por cohorte (cargar fichero existente) ===
auc_path = ROOT / 'data' / '06_interpretacion' / 'robustez' / 'auc_por_cohorte.parquet'

if auc_path.exists():
    df_auc = pd.read_parquet(auc_path)
    print(f"AUC por cohorte cargado: shape={df_auc.shape}")
    print(f"Columnas: {df_auc.columns.tolist()}")
    print()
    print(df_auc.to_string())
    
    # Detectar columna de AUC (puede ser 'auc', 'AUC', 'auc_roc', etc.)
    auc_col = None
    for c in df_auc.columns:
        if 'auc' in c.lower():
            auc_col = c
            break
    
    if auc_col is None:
        print("No se detectó columna AUC. Avisa a Claude.")
    else:
        auc_dict = dict(zip(df_auc['cohorte'].astype(int), df_auc[auc_col].round(4)))
        media = float(np.mean(list(auc_dict.values())))
        std = float(np.std(list(auc_dict.values())))
        
        print(f"\nMedia: {media:.4f}")
        print(f"Std:   {std:.4f}")
        
        ilustracion_19 = {
            'auc_por_cohorte': {int(k): float(v) for k, v in auc_dict.items()},
            'media': round(media, 4),
            'std': round(std, 4),
            'columna_origen': auc_col
        }
        with open(OUTPUT_DIR / 'ilustracion_19_auc_cohorte.json', 'w', encoding='utf-8') as f:
            json.dump(ilustracion_19, f, indent=2, ensure_ascii=False)
        print(f"\nGuardado: {OUTPUT_DIR / 'ilustracion_19_auc_cohorte.json'}")
else:
    print(f"NO encontrado: {auc_path}")

AUC por cohorte cargado: shape=(11, 4)
Columnas: ['cohorte', 'n_alumnos', 'tasa_abandono', 'auc']

    cohorte  n_alumnos  tasa_abandono       auc
0      2010        657       0.377473  0.991857
1      2011        715       0.365035  0.991780
2      2012        680       0.363235  0.981262
3      2013        603       0.374793  0.979871
4      2014        633       0.368088  0.981534
5      2015        580       0.351724  0.961762
6      2016        539       0.309833  0.942679
7      2017        562       0.314947  0.972573
8      2018        538       0.211896  0.924528
9      2019        505       0.142574  0.916795
10     2020        584       0.000000       NaN

Media: nan
Std:   nan

Guardado: c:\FF\AU_UJI_v2\output_para_claude\ilustracion_19_auc_cohorte.json


In [8]:
# === CELDA 8: Ilustración 20 — Curva fiabilidad sobre n=6.596 ===
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

n_bins = 10
prob_true, prob_pred = calibration_curve(y_test_2010, y_proba_2010, n_bins=n_bins, strategy='uniform')
brier = brier_score_loss(y_test_2010, y_proba_2010)

bin_edges = np.linspace(0, 1, n_bins + 1)
ece = 0
n = len(y_test_2010)
y_arr = y_test_2010.values if hasattr(y_test_2010, 'values') else np.asarray(y_test_2010)
for i in range(n_bins):
    if i == n_bins - 1:
        mask = (y_proba_2010 >= bin_edges[i]) & (y_proba_2010 <= bin_edges[i+1])
    else:
        mask = (y_proba_2010 >= bin_edges[i]) & (y_proba_2010 < bin_edges[i+1])
    if mask.sum() > 0:
        bin_acc = y_arr[mask].mean()
        bin_conf = y_proba_2010[mask].mean()
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)

print(f"Curva fiabilidad (n={n}, {n_bins} bins):")
print(f"  prob_pred: {prob_pred.round(4).tolist()}")
print(f"  prob_true: {prob_true.round(4).tolist()}")
print(f"  Brier:     {brier:.4f}")
print(f"  ECE:       {ece:.4f}")

ilustracion_20 = {
    'prob_pred': prob_pred.round(4).tolist(),
    'prob_true': prob_true.round(4).tolist(),
    'brier': round(brier, 4),
    'ece': round(ece, 4),
    'n_bins': n_bins,
    'n_observaciones': n
}
with open(OUTPUT_DIR / 'ilustracion_20_calibracion.json', 'w', encoding='utf-8') as f:
    json.dump(ilustracion_20, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {OUTPUT_DIR / 'ilustracion_20_calibracion.json'}")

Curva fiabilidad (n=6596, 10 bins):
  prob_pred: [0.0311, 0.1417, 0.2448, 0.3489, 0.4472, 0.5475, 0.6515, 0.755, 0.8506, 0.9729]
  prob_true: [0.0244, 0.1315, 0.2088, 0.3656, 0.3967, 0.5354, 0.7236, 0.7928, 0.8868, 0.9754]
  Brier:     0.0702
  ECE:       0.0137

Guardado: c:\FF\AU_UJI_v2\output_para_claude\ilustracion_20_calibracion.json


In [9]:
# === CELDA 9: Resumen ===
print("Ficheros generados en", OUTPUT_DIR)
print()
for f in sorted(OUTPUT_DIR.glob('*.json')):
    print(f"  - {f.name}  ({f.stat().st_size} bytes)")
print()
print("SIGUIENTE PASO: sube los 4 ficheros JSON al chat con Claude.")

Ficheros generados en c:\FF\AU_UJI_v2\output_para_claude

  - features_dict_legible.json  (1148 bytes)
  - ilustracion_17_shap_top15.json  (618 bytes)
  - ilustracion_19_auc_cohorte.json  (320 bytes)
  - ilustracion_20_calibracion.json  (391 bytes)

SIGUIENTE PASO: sube los 4 ficheros JSON al chat con Claude.
